# Chapter 3 — Inner Products and Euclidean Affine Spaces

Companion notebook to Chapter 3 of *Advanced Calculus for Machine Learning*.
It checks the numerical claims of the chapter with PyTorch. Everything runs
on a CPU in a few seconds.

Conventions: vectors are columns, covectors are rows, $M$ and $N$ are Gram
matrices, $A^\top$ is the dual operator and $A^*$ the adjoint.

Contents:

1. Inner products and the Cauchy–Schwarz inequality
2. Gram–Schmidt and Cholesky (Exercise 3.1)
3. The Riesz map: one covector, several representatives
4. The basis test, passed
5. The adjoint is $M^{-1}A^\top N$
6. Adjoint of the derivative; self-adjointness depends on the inner product
7. Least squares, ordinary and weighted (Exercise 3.6)
8. Which solution gradient descent finds
9. Distance to a hyperplane, Mahalanobis distance, isometries

In [1]:
import math
import torch

torch.set_default_dtype(torch.float64)
torch.manual_seed(0)


def random_spd(n):
    B = torch.randn(n, n)
    return B @ B.T + n * torch.eye(n)


def inner(u, v, M):
    return u @ M @ v


print(torch.__version__)

2.15.0+cu132


## 1. Inner products and the Cauchy–Schwarz inequality

An inner product on $\mathbb R^n$ is $\langle u, v\rangle_M = u^\top Mv$ with
$M$ symmetric positive definite. Under a change of basis its Gram matrix
becomes $P^\top MP$, and the number $\langle u, v\rangle$ does not change.

In [2]:
n = 5
M = random_spd(n)
worst = 0.0
for _ in range(1000):
    u, v = torch.randn(n), torch.randn(n)
    ratio = inner(u, v, M).abs() / (inner(u, u, M).sqrt() * inner(v, v, M).sqrt())
    worst = max(worst, ratio.item())
print("largest |<u,v>| / (|u| |v|) over 1000 random pairs:", round(worst, 4), "<= 1")

P = torch.randn(n, n)
u, v = torch.randn(n), torch.randn(n)
u_new, v_new = torch.linalg.solve(P, u), torch.linalg.solve(P, v)
assert torch.allclose(inner(u_new, v_new, P.T @ M @ P), inner(u, v, M))
print("the inner product is the same number in both bases")

largest |<u,v>| / (|u| |v|) over 1000 random pairs: 0.9864 <= 1
the inner product is the same number in both bases


## 2. Gram–Schmidt and Cholesky

Exercise 3.1: Gram–Schmidt on the standard basis for
$M = \begin{bmatrix}2&1\\1&2\end{bmatrix}$ gives an upper triangular $P$
with $P^\top MP = I$, and $L = P^{-\top}$ is the Cholesky factor of $M$.

In [3]:
def gram_schmidt(V, M):
    # columns of V: a basis; returns the columns of an M-orthonormal basis
    E = []
    for k in range(V.shape[1]):
        u = V[:, k].clone()
        for e in E:
            u = u - inner(e, V[:, k], M) * e
        E.append(u / inner(u, u, M).sqrt())
    return torch.stack(E, dim=1)


M2 = torch.tensor([[2.0, 1.0], [1.0, 2.0]])
P = gram_schmidt(torch.eye(2), M2)
print("P =\n", P)
expected_P = torch.tensor([[1 / math.sqrt(2), -1 / math.sqrt(6)], [0.0, 2 / math.sqrt(6)]])
assert torch.allclose(P, expected_P)
assert torch.allclose(P.T @ M2 @ P, torch.eye(2))

L = torch.linalg.inv(P).T
print("L = P^{-T} =\n", L)
assert torch.allclose(L, torch.linalg.cholesky(M2))
assert torch.allclose(L @ L.T, M2)
print("L is the Cholesky factor returned by torch.linalg.cholesky")

P =
 tensor([[ 0.7071, -0.4082],
        [ 0.0000,  0.8165]])
L = P^{-T} =
 tensor([[1.4142, 0.0000],
        [0.7071, 1.2247]])
L is the Cholesky factor returned by torch.linalg.cholesky


## 3. The Riesz map

Theorem 3.10: the vector that represents the covector $\phi$ has the
column $M^{-1}[\phi]^\top$, and the dual norm is
$\lVert\phi\rVert_*^2 = [\phi]M^{-1}[\phi]^\top$ (Example 3.13,
Exercise 3.2).

In [4]:
phi = torch.tensor([2.0, 1.0])               # a row
v = torch.tensor([1.0, 1.0])
metrics = {
    "dot product      ": torch.eye(2),
    "M = diag(1/4, 1) ": torch.diag(torch.tensor([0.25, 1.0])),
    "M = [[2,1],[1,2]]": M2,
}
for name, M in metrics.items():
    u = torch.linalg.solve(M, phi)           # Riesz representative
    dual_norm = (phi @ u).sqrt()
    assert torch.allclose(inner(u, v, M), phi @ v)        # <u, v> = phi(v)
    print(f"{name}: representative {[round(c, 4) + 0.0 for c in u.tolist()]}, "
          f"dual norm^2 = {(phi @ u).item():.4f}")

dot product      : representative [2.0, 1.0], dual norm^2 = 5.0000
M = diag(1/4, 1) : representative [8.0, 1.0], dual norm^2 = 17.0000
M = [[2,1],[1,2]]: representative [1.0, 0.0], dual norm^2 = 2.0000


Proposition 3.11: on the unit ball, $\phi$ is largest at $u/\lVert u\rVert$,
where it equals $\lVert u\rVert = \lVert\phi\rVert_*$. We sample the unit
sphere of $M = \mathrm{diag}(\tfrac14, 1)$ and compare.

In [5]:
M = torch.diag(torch.tensor([0.25, 1.0]))
angles = torch.linspace(0, 2 * math.pi, 200001)
circle = torch.stack([2 * torch.cos(angles), torch.sin(angles)], dim=1)   # x^2/4 + y^2 = 1
values = circle @ phi
best = circle[values.argmax()]
u = torch.linalg.solve(M, phi)
print("maximum of phi on the unit sphere:", round(values.max().item(), 6),
      "  sqrt(17) =", round(math.sqrt(17), 6))
print("attained at", [round(c, 4) for c in best.tolist()],
      "  u/|u| =", [round(c, 4) for c in (u / inner(u, u, M).sqrt()).tolist()])
assert abs(values.max().item() - math.sqrt(17)) < 1e-8

maximum of phi on the unit sphere: 4.123106   sqrt(17) = 4.123106
attained at [1.9403, 0.2425]   u/|u| = [1.9403, 0.2425]


## 4. The basis test, passed

Proposition 3.14: with $x' = P^{-1}x$, $\gamma' = P^\top\gamma$ and
$M' = P^\top MP$, the update $x - \eta M^{-1}\gamma$ gives the same point in
every basis. The partial derivatives are computed by autograd in each
coordinate system.

In [6]:
def loss_fn(x):
    return torch.log(1 + torch.exp(x[0] - 2 * x[1])) + 0.3 * (x ** 2).sum() + x[0] * x[1]


def metric_step(x0, M, P, eta=0.1):
    # one step of x - eta M^{-1} gamma, computed in the basis P; result in old coordinates
    xp = torch.linalg.solve(P, x0).clone().requires_grad_(True)
    loss_fn(P @ xp).backward()
    M_new = P.T @ M @ P
    return P @ (xp.detach() - eta * torch.linalg.solve(M_new, xp.grad))


x0 = torch.tensor([0.7, -0.4])
M = random_spd(2)
reference = metric_step(x0, M, torch.eye(2))
for _ in range(5):
    P = torch.randn(2, 2)
    assert torch.allclose(metric_step(x0, M, P), reference)
print("x - eta M^{-1} gamma: same point in 5 random bases")

# Exercise 3.7 / Example 2.26: gamma = (2, 1), M = I in e, P = diag(2, 1)
gamma = torch.tensor([2.0, 1.0])
P = torch.diag(torch.tensor([2.0, 1.0]))
M_new, gamma_new = P.T @ torch.eye(2) @ P, P.T @ gamma
step_metric = P @ torch.linalg.solve(M_new, gamma_new)
step_plain = P @ gamma_new                      # uses M' = I: another inner product
print("with M' = P^T M P:", step_metric.tolist(), "   with M' = I:", step_plain.tolist())
assert torch.equal(step_metric, torch.tensor([2.0, 1.0]))
assert torch.equal(step_plain, torch.tensor([8.0, 1.0]))

x - eta M^{-1} gamma: same point in 5 random bases
with M' = P^T M P: [2.0, 1.0]    with M' = I: [8.0, 1.0]


## 5. The adjoint is $M^{-1}A^\top N$

Theorem 3.16, checked on random maps and inner products, with the rules of
Proposition 3.18. The matrix of the dual operator is $A^\top$ whatever $M$
and $N$ are.

In [7]:
def adjoint(A, M, N):
    return torch.linalg.solve(M, A.T @ N)


n, m, k = 4, 3, 2
M, N, K = random_spd(n), random_spd(m), random_spd(k)
A, B = torch.randn(m, n), torch.randn(k, m)
A_star = adjoint(A, M, N)
v, w = torch.randn(n), torch.randn(m)
assert torch.allclose(inner(A @ v, w, N), inner(v, A_star @ w, M))
print("<Av, w>_N = <v, A* w>_M")
assert torch.allclose(adjoint(B @ A, M, K), A_star @ adjoint(B, N, K))
print("(BA)* = A* B*")
assert torch.allclose(adjoint(A_star, N, M), A)
print("(A*)* = A")
assert torch.allclose(inner(v, A_star @ A @ v, M), inner(A @ v, A @ v, N))
print("<v, A*A v> = |Av|^2")
print("A* equals A^T here?", torch.allclose(A_star, A.T))

# Example 3.17
A2 = torch.tensor([[1.0, 2.0], [0.0, 1.0]])
A2_star = adjoint(A2, torch.diag(torch.tensor([1.0, 4.0])), torch.eye(2))
print("Example 3.17, [A*] =\n", A2_star)
assert torch.equal(A2_star, torch.tensor([[1.0, 0.0], [0.5, 0.25]]))

<Av, w>_N = <v, A* w>_M
(BA)* = A* B*
(A*)* = A
<v, A*A v> = |Av|^2
A* equals A^T here? False
Example 3.17, [A*] =
 tensor([[1.0000, 0.0000],
        [0.5000, 0.2500]])


Theorem 3.22: $\ker A^* = (\operatorname{im}A)^\perp$, with orthogonality
measured by the inner product of the target space.

In [8]:
m, n = 4, 2
A = torch.randn(m, n)
M, N = random_spd(n), random_spd(m)
A_star = adjoint(A, M, N)
# a basis of ker A* (the SVD is only a convenient way to get one)
_, _, Vh = torch.linalg.svd(A_star)
kernel = Vh[n:].T                               # m x (m - n)
assert torch.allclose(A_star @ kernel, torch.zeros(n, m - n), atol=1e-12)
gram = kernel.T @ N @ A                         # <k_i, A e_j>_N
print("inner products between ker A* and im A:\n", gram.abs().max().item())
assert gram.abs().max() < 1e-12

inner products between ker A* and im A:
 5.0605009783089685e-15


## 6. Adjoint of the derivative; self-adjointness

Exercise 3.3: on polynomials of degree at most 1 with
$\langle p, q\rangle = \int_{-1}^1pq$, the adjoint of $D$ sends $1$ to $3t$
and $t$ to $0$. It is not $-D$.

In [9]:
M_poly = torch.tensor([[2.0, 0.0], [0.0, 2.0 / 3.0]])     # Gram matrix in the basis (1, t)
D = torch.tensor([[0.0, 1.0], [0.0, 0.0]])
D_star = adjoint(D, M_poly, M_poly)
print("[D*] =\n", D_star)
assert torch.allclose(D_star, torch.tensor([[0.0, 0.0], [3.0, 0.0]]))

[D*] =
 tensor([[0., 0.],
        [3., 0.]])


Exercise 3.4: the operator with matrix $\begin{bmatrix}1&1\\0&2\end{bmatrix}$
is not self-adjoint for the dot product, and is self-adjoint for
$M = \begin{bmatrix}1&-1\\-1&2\end{bmatrix}$.

In [10]:
A = torch.tensor([[1.0, 1.0], [0.0, 2.0]])
M_sa = torch.tensor([[1.0, -1.0], [-1.0, 2.0]])
print("self-adjoint for the dot product:", torch.allclose(adjoint(A, torch.eye(2), torch.eye(2)), A))
print("self-adjoint for M              :", torch.allclose(adjoint(A, M_sa, M_sa), A))
v1, v2 = torch.tensor([1.0, 0.0]), torch.tensor([1.0, 1.0])   # eigenvectors of A
print("<v1, v2>_M =", inner(v1, v2, M_sa).item(), "   v1 . v2 =", (v1 @ v2).item())

self-adjoint for the dot product: False
self-adjoint for M              : True
<v1, v2>_M = 0.0    v1 . v2 = 1.0


## 7. Least squares

Theorem 3.24: the normal equations are $A^\top NA\,x = A^\top Nb$. The inner
product $N$ of the target space weights the equations; the inner product of
the source space does not appear. Exercise 3.6 fits a line to three points.

In [11]:
A = torch.tensor([[1.0, 0.0], [1.0, 1.0], [1.0, 2.0]])
b = torch.tensor([1.0, 2.0, 2.0])


def least_squares(A, b, N):
    return torch.linalg.solve(A.T @ N @ A, A.T @ N @ b)


x_plain = least_squares(A, b, torch.eye(3))
N = torch.diag(torch.tensor([1.0, 1.0, 4.0]))
x_weighted = least_squares(A, b, N)
print("ordinary:", x_plain.tolist(), "  expected", [7 / 6, 1 / 2])
print("weighted:", x_weighted.tolist(), "  expected", [25 / 21, 3 / 7])
assert torch.allclose(x_plain, torch.tensor([7 / 6, 1 / 2]))
assert torch.allclose(x_weighted, torch.tensor([25 / 21, 3 / 7]))
assert torch.allclose(x_plain, torch.linalg.lstsq(A, b.unsqueeze(1)).solution.squeeze(1))

# the residual is orthogonal to im A for the inner product N
residual = A @ x_weighted - b
print("A^T N (Ax - b) =", (A.T @ N @ residual).tolist())

# Exercise 3.5: A (A*A)^{-1} A* is the orthogonal projection onto im A
M_src = random_spd(2)
A_star = adjoint(A, M_src, N)
Pi = A @ torch.linalg.solve(A_star @ A, A_star)
assert torch.allclose(Pi @ Pi, Pi) and torch.allclose(adjoint(Pi, N, N), Pi)
assert torch.allclose(Pi @ b, A @ x_weighted)
print("projection: idempotent, self-adjoint for N, and Pi b = A x (whatever M is)")

ordinary: [1.1666666666666665, 0.5]   expected [1.1666666666666667, 0.5]
weighted: [1.1904761904761907, 0.42857142857142844]   expected [1.1904761904761905, 0.42857142857142855]
A^T N (Ax - b) = [-4.440892098500626e-16, -1.3322676295501878e-15]


projection: idempotent, self-adjoint for N, and Pi b = A x (whatever M is)


## 8. Which solution gradient descent finds

One equation, two unknowns: $\theta_1 + \theta_2 = 2$ (Exercise 3.8,
Figure 3.3). The update $x - \eta M^{-1}\gamma$ is run with autograd from
$x_0 = (0.5, -0.5)$ for two inner products. Theorem 3.25 predicts the
limit: the solution closest to $x_0$ for $M$, namely
$x_0 + X^*(XX^*)^{-1}(y - Xx_0)$ with $X^* = M^{-1}X^\top$.

In [12]:
def metric_gd(X, y, x0, M, eta, steps):
    x = x0.clone().requires_grad_(True)
    for _ in range(steps):
        loss = 0.5 * ((X @ x - y) ** 2).sum()
        (g,) = torch.autograd.grad(loss, x)
        with torch.no_grad():
            x -= eta * torch.linalg.solve(M, g)
    return x.detach()


def closest_solution(X, y, x0, M):
    X_star = torch.linalg.solve(M, X.T)
    return x0 + X_star @ torch.linalg.solve(X @ X_star, y - X @ x0)


X = torch.tensor([[1.0, 1.0]])
y = torch.tensor([2.0])
x0 = torch.tensor([0.5, -0.5])
for name, M in [("dot product    ", torch.eye(2)),
                ("M = diag(1, 4) ", torch.diag(torch.tensor([1.0, 4.0])))]:
    limit = metric_gd(X, y, x0, M, eta=0.3, steps=200)
    predicted = closest_solution(X, y, x0, M)
    print(f"{name}: gradient descent -> {[round(c, 6) for c in limit.tolist()]}, "
          f"closest solution {[round(c, 6) for c in predicted.tolist()]}")
    assert torch.allclose(limit, predicted)

dot product    : gradient descent -> [1.5, 0.5], closest solution [1.5, 0.5]
M = diag(1, 4) : gradient descent -> [2.1, -0.1], closest solution [2.1, -0.1]


The same on a random problem with 3 equations and 8 unknowns, and a check
that the limit is closer to the starting point (for $M$) than other
solutions.

In [13]:
n_data, p = 3, 8
X, y = torch.randn(n_data, p), torch.randn(n_data)
M, x0 = random_spd(p), torch.randn(p)
eta = 1.0 / torch.linalg.eigvalsh(X @ torch.linalg.solve(M, X.T)).max().item()
limit = metric_gd(X, y, x0, M, eta=eta, steps=5000)
predicted = closest_solution(X, y, x0, M)
assert torch.allclose(X @ limit, y, atol=1e-9) and torch.allclose(limit, predicted, atol=1e-8)

_, _, Vh = torch.linalg.svd(X)
kernel = Vh[n_data:].T
dist = lambda x: inner(x - x0, x - x0, M).sqrt().item()
others = [dist(predicted + kernel @ torch.randn(p - n_data)) for _ in range(1000)]
print("distance from x0 to the limit:", round(dist(limit), 4),
      "   smallest distance to 1000 other solutions:", round(min(others), 4))
assert dist(limit) <= min(others)

distance from x0 to the limit: 5.4053    smallest distance to 1000 other solutions: 5.5128


## 9. Distances

Proposition 3.28 and Exercise 3.9: the distance from a point to the
hyperplane $\{\ell = c\}$ is $|\ell(a) - c|/\lVert\phi\rVert_*$. Here
$H = \{3x_1 + 4x_2 = 5\}$ and $a = (0, 0)$.

In [14]:
phi, c, a = torch.tensor([3.0, 4.0]), 5.0, torch.zeros(2)
for name, M in [("dot product   ", torch.eye(2)),
                ("M = diag(1, 4)", torch.diag(torch.tensor([1.0, 4.0])))]:
    u = torch.linalg.solve(M, phi)
    dual_sq = phi @ u
    distance = abs((phi @ a).item() - c) / dual_sq.sqrt().item()
    closest = a - ((phi @ a) - c) / dual_sq * u
    assert torch.allclose(phi @ closest, torch.tensor(c))
    assert abs(inner(closest - a, closest - a, M).sqrt().item() - distance) < 1e-12
    # brute force along the line H
    t = torch.linspace(-10, 10, 2000001)
    line = torch.stack([t, (c - 3 * t) / 4], dim=1)
    brute = ((line @ M) * line).sum(dim=1).sqrt().min().item()
    print(f"{name}: distance {distance:.6f} (brute force {brute:.6f}), "
          f"closest point {[round(v, 6) for v in closest.tolist()]}")

dot product   : distance 1.000000 (brute force 1.000000), closest point [0.6, 0.8]
M = diag(1, 4): distance 1.386750 (brute force 1.386750), closest point [1.153846, 0.384615]


Mahalanobis distance and whitening: with $\Sigma = LL^\top$, the change of
frame $z = L^{-1}(x - \mu)$ turns the Mahalanobis distance into the ordinary
one.

In [15]:
d = 4
Sigma, mu = random_spd(d), torch.randn(d)
L = torch.linalg.cholesky(Sigma)
x1, x2 = torch.randn(d), torch.randn(d)
mahalanobis = inner(x1 - x2, x1 - x2, torch.linalg.inv(Sigma)).sqrt()
z1 = torch.linalg.solve_triangular(L, (x1 - mu).unsqueeze(1), upper=False).squeeze(1)
z2 = torch.linalg.solve_triangular(L, (x2 - mu).unsqueeze(1), upper=False).squeeze(1)
print("Mahalanobis distance:", mahalanobis.item(), "  ordinary distance after whitening:",
      (z1 - z2).norm().item())
assert torch.allclose(mahalanobis, (z1 - z2).norm())

Mahalanobis distance: 0.6537809530805863   ordinary distance after whitening: 0.653780953080586


Proposition 3.27: an affine map preserves distances exactly when its linear
part satisfies $A^*A = \mathrm{id}$, that is, $A^\top MA = M$. A matrix that
is orthogonal for the dot product is in general not an isometry for another
inner product.

In [16]:
M = random_spd(3)
Q, _ = torch.linalg.qr(torch.randn(3, 3))                 # Q^T Q = I
print("Q* Q = I for the dot product:", torch.allclose(adjoint(Q, torch.eye(3), torch.eye(3)) @ Q, torch.eye(3)))
print("Q* Q = I for M              :", torch.allclose(adjoint(Q, M, M) @ Q, torch.eye(3)))

# an isometry for M: conjugate Q by a basis that is orthonormal for M
Lm = torch.linalg.cholesky(M)                             # M = Lm Lm^T
A_iso = torch.linalg.solve(Lm.T, Q @ Lm.T)                # Lm^{-T} Q Lm^T
assert torch.allclose(adjoint(A_iso, M, M) @ A_iso, torch.eye(3))
a, b, shift = torch.randn(3), torch.randn(3), torch.randn(3)
f = lambda x: A_iso @ x + shift
d_before = inner(b - a, b - a, M).sqrt()
d_after = inner(f(b) - f(a), f(b) - f(a), M).sqrt()
print("distance before and after the affine isometry:", d_before.item(), d_after.item())
assert torch.allclose(d_before, d_after)

Q* Q = I for the dot product: True
Q* Q = I for M              : False
distance before and after the affine isometry: 4.837415578770551 4.8374155787705515
